In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [4]:
history = pd.read_csv("history_clean.csv")
flight = pd.read_csv("flight_clean.csv")
calendar = pd.read_csv("calendar_clean.csv")

In [5]:
calendar.head()

,Date,Start of Year,Start of Quarter,Start of Month
0,2012-01-01,2012-01-01,2012-01-01,2012-01-01
1,2012-01-02,2012-01-01,2012-01-01,2012-01-01
2,2012-01-03,2012-01-01,2012-01-01,2012-01-01
3,2012-01-04,2012-01-01,2012-01-01,2012-01-01
4,2012-01-05,2012-01-01,2012-01-01,2012-01-01


In [6]:
flight.shape

(391014, 8)

In [7]:
history.shape

(16737, 16)

How many flights has each customer taken during the analysis period?

In [8]:
customer_features = flight.groupby("Loyalty Number").agg({
    "Total Flights":"sum"
})

customer_features.head()

,Total Flights
Loyalty Number,
100018,46
100102,51
100140,47
100214,22
100272,37


In [9]:
customer_features.head()

,Total Flights
Loyalty Number,
100018,46
100102,51
100140,47
100214,22
100272,37


How much total distance has each customer travelled?

In [10]:
customer_features['Total Distance']=flight.groupby("Loyalty Number")["Distance"].sum()

In [11]:
customer_features.head()

,Total Flights,Total Distance
Loyalty Number,,
100018,46,81190
100102,51,68918
100140,47,72856
100214,22,38236
100272,37,54997


How many loyalty points has each customer earned?

In [12]:
customer_features["Total Points Earned"] = (
    flight.groupby("Loyalty Number")["Points Accumulated"].sum()
)

customer_features.head()

,Total Flights,Total Distance,Total Points Earned
Loyalty Number,,,
100018,46,81190,81190.0
100102,51,68918,68918.0
100140,47,72856,72856.0
100214,22,38236,38236.0
100272,37,54997,54997.0


How many loyalty points has each customer redeemed?

In [13]:
customer_features["Total Points Redeemed"] = (
    flight.groupby("Loyalty Number")["Points Redeemed"].sum()
)

customer_features.head()

,Total Flights,Total Distance,Total Points Earned,Total Points Redeemed
Loyalty Number,,,,
100018,46,81190,81190.0,1513
100102,51,68918,68918.0,1195
100140,47,72856,72856.0,593
100214,22,38236,38236.0,861
100272,37,54997,54997.0,1007


In how many months was the customer actually active?

In [14]:
active_flights = flight[flight["Total Flights"] > 0]

In [15]:
customer_features["Active Months"] = (
    active_flights.groupby("Loyalty Number")["Month"]
    .nunique()
)

In [16]:
customer_features.head()

,Total Flights,Total Distance,Total Points Earned,Total Points Redeemed,Active Months
Loyalty Number,,,,,
100018,46,81190,81190.0,1513,11.0
100102,51,68918,68918.0,1195,12.0
100140,47,72856,72856.0,593,11.0
100214,22,38236,38236.0,861,7.0
100272,37,54997,54997.0,1007,11.0


Average Flights per Active Month

In [17]:
customer_features["Avg Flights per Active Month"] = (
    customer_features["Total Flights"] /
    customer_features["Active Months"]
)

customer_features.head()

,Total Flights,Total Distance,Total Points Earned,Total Points Redeemed,Active Months,Avg Flights per Active Month
Loyalty Number,,,,,,
100018,46,81190,81190.0,1513,11.0,4.181818
100102,51,68918,68918.0,1195,12.0,4.250000
100140,47,72856,72856.0,593,11.0,4.272727
100214,22,38236,38236.0,861,7.0,3.142857
100272,37,54997,54997.0,1007,11.0,3.363636


How actively does the customer use the loyalty program?

In [18]:
customer_features["Redemption Ratio"] = np.where(
    customer_features["Total Points Earned"] > 0,
    customer_features["Total Points Redeemed"] /
    customer_features["Total Points Earned"],
    0
)

In [19]:
customer_features.head()


,Total Flights,Total Distance,Total Points Earned,Total Points Redeemed,Active Months,Avg Flights per Active Month,Redemption Ratio
Loyalty Number,,,,,,,
100018,46,81190,81190.0,1513,11.0,4.181818,0.018635
100102,51,68918,68918.0,1195,12.0,4.250000,0.017339
100140,47,72856,72856.0,593,11.0,4.272727,0.008139
100214,22,38236,38236.0,861,7.0,3.142857,0.022518
100272,37,54997,54997.0,1007,11.0,3.363636,0.018310


Does the customer usually take short trips or long trips?

In [20]:
customer_features["Avg Distance per Flight"] = np.where(
    customer_features["Total Flights"] > 0,
    customer_features["Total Distance"] /
    customer_features["Total Flights"],
    0
)

customer_features.head()

,Total Flights,Total Distance,Total Points Earned,Total Points Redeemed,Active Months,Avg Flights per Active Month,Redemption Ratio,Avg Distance per Flight
Loyalty Number,,,,,,,,
100018,46,81190,81190.0,1513,11.0,4.181818,0.018635,1765.000000
100102,51,68918,68918.0,1195,12.0,4.250000,0.017339,1351.333333
100140,47,72856,72856.0,593,11.0,4.272727,0.008139,1550.127660
100214,22,38236,38236.0,861,7.0,3.142857,0.022518,1738.000000
100272,37,54997,54997.0,1007,11.0,3.363636,0.018310,1486.405405


How long has the customer been a loyalty member?

In [21]:
customer_features = customer_features.merge(
    history[
        ["Loyalty Number",
         "Enrollment Year",
         "Enrollment Month"]
    ],
    on="Loyalty Number",
    how="left"
)

customer_features.head()

,Loyalty Number,Total Flights,Total Distance,Total Points Earned,Total Points Redeemed,Active Months,Avg Flights per Active Month,Redemption Ratio,Avg Distance per Flight,Enrollment Year,Enrollment Month
0,100018,46,81190,81190.0,1513,11.0,4.181818,0.018635,1765.000000,2016,8
1,100102,51,68918,68918.0,1195,12.0,4.250000,0.017339,1351.333333,2013,3
2,100140,47,72856,72856.0,593,11.0,4.272727,0.008139,1550.127660,2016,7
3,100214,22,38236,38236.0,861,7.0,3.142857,0.022518,1738.000000,2015,8
4,100272,37,54997,54997.0,1007,11.0,3.363636,0.018310,1486.405405,2014,1


How long has the customer been part of the airline's loyalty program?

In [22]:
customer_features["Loyalty Tenure (Months)"] = (
    (2018 - customer_features["Enrollment Year"]) * 12
    + (12 - customer_features["Enrollment Month"])
)

customer_features.head()

,Loyalty Number,Total Flights,Total Distance,Total Points Earned,Total Points Redeemed,Active Months,Avg Flights per Active Month,Redemption Ratio,Avg Distance per Flight,Enrollment Year,Enrollment Month,Loyalty Tenure (Months)
0,100018,46,81190,81190.0,1513,11.0,4.181818,0.018635,1765.000000,2016,8,28
1,100102,51,68918,68918.0,1195,12.0,4.250000,0.017339,1351.333333,2013,3,69
2,100140,47,72856,72856.0,593,11.0,4.272727,0.008139,1550.127660,2016,7,29
3,100214,22,38236,38236.0,861,7.0,3.142857,0.022518,1738.000000,2015,8,40
4,100272,37,54997,54997.0,1007,11.0,3.363636,0.018310,1486.405405,2014,1,59


In [23]:
customer_features.head()

,Loyalty Number,Total Flights,Total Distance,Total Points Earned,Total Points Redeemed,Active Months,Avg Flights per Active Month,Redemption Ratio,Avg Distance per Flight,Enrollment Year,Enrollment Month,Loyalty Tenure (Months)
0,100018,46,81190,81190.0,1513,11.0,4.181818,0.018635,1765.000000,2016,8,28
1,100102,51,68918,68918.0,1195,12.0,4.250000,0.017339,1351.333333,2013,3,69
2,100140,47,72856,72856.0,593,11.0,4.272727,0.008139,1550.127660,2016,7,29
3,100214,22,38236,38236.0,861,7.0,3.142857,0.022518,1738.000000,2015,8,40
4,100272,37,54997,54997.0,1007,11.0,3.363636,0.018310,1486.405405,2014,1,59


Last Flight Date

In [24]:
active_flights = flight[flight["Total Flights"] > 0].copy()

In [25]:
active_flights["Flight Date"] = pd.to_datetime(
    active_flights["Year"].astype(str) + "-" +
    active_flights["Month"].astype(str) + "-01"
)

In [26]:
last_flight = (
    active_flights
    .groupby("Loyalty Number")["Flight Date"]
    .max()
)
last_flight

,Flight Date
Loyalty Number,
100018,2018-12-01
100102,2018-12-01
100140,2018-11-01
100214,2018-12-01
100272,2018-11-01
...,...
999788,2017-10-01
999902,2018-10-01
999940,2018-12-01


In [27]:
customer_features = customer_features.merge(
    last_flight.rename("Last Flight Date"),
    on="Loyalty Number",
    how="left"
)

In [28]:
reference_date = pd.Timestamp("2018-12-01")

customer_features["Months Since Last Flight"] = (
    (reference_date.year - customer_features["Last Flight Date"].dt.year) * 12
    + (reference_date.month - customer_features["Last Flight Date"].dt.month)
)

In [29]:
customer_features.head()

,Loyalty Number,Total Flights,Total Distance,Total Points Earned,Total Points Redeemed,Active Months,Avg Flights per Active Month,Redemption Ratio,Avg Distance per Flight,Enrollment Year,Enrollment Month,Loyalty Tenure (Months),Last Flight Date,Months Since Last Flight
0,100018,46,81190,81190.0,1513,11.0,4.181818,0.018635,1765.000000,2016,8,28,2018-12-01,0.0
1,100102,51,68918,68918.0,1195,12.0,4.250000,0.017339,1351.333333,2013,3,69,2018-12-01,0.0
2,100140,47,72856,72856.0,593,11.0,4.272727,0.008139,1550.127660,2016,7,29,2018-11-01,1.0
3,100214,22,38236,38236.0,861,7.0,3.142857,0.022518,1738.000000,2015,8,40,2018-12-01,0.0
4,100272,37,54997,54997.0,1007,11.0,3.363636,0.018310,1486.405405,2014,1,59,2018-11-01,1.0


Churn Definition: A customer is classified as churned if they have not taken any flights for 6 or more months by the end of the observation period. Six months was selected as a balanced threshold because it reduces the likelihood of misclassifying seasonal travelers while still allowing the airline sufficient time to identify disengaging customers and launch retention campaigns before they are permanently lost.


In [30]:
customer_features["Churn"] = np.where(
    customer_features["Months Since Last Flight"] >= 6,
    1,
    0
)

In [31]:
customer_features.head()

,Loyalty Number,Total Flights,Total Distance,Total Points Earned,Total Points Redeemed,Active Months,Avg Flights per Active Month,Redemption Ratio,Avg Distance per Flight,Enrollment Year,Enrollment Month,Loyalty Tenure (Months),Last Flight Date,Months Since Last Flight,Churn
0,100018,46,81190,81190.0,1513,11.0,4.181818,0.018635,1765.000000,2016,8,28,2018-12-01,0.0,0
1,100102,51,68918,68918.0,1195,12.0,4.250000,0.017339,1351.333333,2013,3,69,2018-12-01,0.0,0
2,100140,47,72856,72856.0,593,11.0,4.272727,0.008139,1550.127660,2016,7,29,2018-11-01,1.0,0
3,100214,22,38236,38236.0,861,7.0,3.142857,0.022518,1738.000000,2015,8,40,2018-12-01,0.0,0
4,100272,37,54997,54997.0,1007,11.0,3.363636,0.018310,1486.405405,2014,1,59,2018-11-01,1.0,0


In [32]:
customer_features["Churn"].value_counts()

,count
Churn,
0,15838
1,899


In [33]:
customer_features["Churn"].value_counts(normalize=True) * 100

,proportion
Churn,
0,94.628667
1,5.371333


Instead of Churn = Months Since Last Flight >= 6 we will use Churn = Cancellation Year is not null To prevent data leakage

In [34]:
history["Cancellation Year"].notna().sum()

np.int64(2067)

In [35]:
history["Cancellation Year"].isna().sum()

np.int64(14670)

In [36]:
history["Cancellation Year"].value_counts(dropna=False)

,count
Cancellation Year,
NaN,14670
2018.0,645
2017.0,506
2016.0,427
2015.0,265
2014.0,181
2013.0,43


In [37]:
history["Churn"] = np.where(
    history["Cancellation Year"].notna(),
    1,
    0
)

In [38]:
history["Churn"].value_counts()

,count
Churn,
0,14670
1,2067


In [39]:
customer_features = customer_features.drop(columns=["Churn"])

customer_features = customer_features.merge(
    history[["Loyalty Number", "Churn"]],
    on="Loyalty Number",
    how="left"
)


In [43]:
customer_features["At Risk"] = np.where(
    customer_features["Months Since Last Flight"] >= 6,
    1,
    0
)

In [44]:
customer_features.to_csv(
    "customer_features.csv",
    index=False
)